In [ ]:
# Definimos las reglas para etiquetar las anomalías
# Usamos las variables duración, velocidad, tarifa/km creadas en la sección de ingeniería de características
df_anomalias = df_features.withColumn(
    "tipo_anomalia",
    when((col("avg_speed_mph") > 75) & (col("trip_distance") > 2), "Teletransporte (Velocidad Imposible)")
    .when((col("fare_per_km") > 40) & (col("trip_distance_km") < 1.5) & (col("trip_duration_minutes") < 5), "Posible Fraude (Tarifa altísima, viaje súper corto)")
    .when((col("trip_duration_minutes") > 120) & (col("trip_distance") < 2), "Atasco Infinito (Más de 2 horas para menos de 2 millas)")
    .otherwise("Viaje Normal")
)

# Filtramos el DataFrame para quedarnos solo con los viajes anómalos
solo_anomalias = df_anomalias.filter(col("tipo_anomalia") != "Viaje Normal")

# Contamos cuántas anomalías de cada tipo han sido detectadas
resumen_anomalias = solo_anomalias.groupBy("tipo_anomalia").count().orderBy(col("count").desc())

print("Resumen de la cantidad de anomalías detectadas:")
resumen_anomalias.show(truncate=False)

print("Muestra de los registros anómalos:")
solo_anomalias.select(
    "tipo_anomalia", 
    "trip_distance_km", 
    "trip_duration_minutes", 
    "avg_speed_mph", 
    "fare_amount", 
    "fare_per_km"
).show(15, truncate=False)